In [ ]:
# locate the project root
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [ ]:
from pathlib import Path
from time import perf_counter

import joblib
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from src.config import (
    RANDOM_STATE,
    PROCESSED_DATA_DIR,
    SENTIMENT_ARTIFACTS_DIR,
    REPORTS_DIR,
    ARTIFACTS_DIR,
    PLOTS_DIR,
)
from src.data.load import load_csv
from src.data.evaluation import (
    evaluate_model,
    get_top_features_per_class,
    display_top_features_per_class,
    create_model_result_row,
)
from src.data.validate import validate_columns

In [ ]:
# Load processed dataset
df = load_csv(PROCESSED_DATA_DIR)
df.head()

In [ ]:
# validate the required columns: text and rating
validate_columns(df.columns)

In [ ]:
# remove invalid rows
model_df = (
    df[["classical_text", "sentiment_label"]]
    .dropna()
    .copy()
)

model_df["classical_text"] = (
    model_df["classical_text"]
    .astype(str)
    .str.strip()
)

model_df = model_df[
    model_df["classical_text"].str.len() > 0
].reset_index(drop=True)

print(f"Model dataset shape: {model_df.shape}")

In [ ]:
# inspect class imbalance
class_counts = model_df["sentiment_label"].value_counts()
class_distribution = model_df["sentiment_label"].value_counts(
    normalize=True
).mul(100)

class_summary = pd.DataFrame(
    {
        "count": class_counts,
        "percentage": class_distribution.round(2),
    }
)

display(class_summary)

In [ ]:
# plot the model
class_counts.plot(kind="bar")

plt.title("Sentiment Class Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Stratified train, validation and test split
# The evaluation sets stay naturally imbalanced.

X = model_df["classical_text"]
y = model_df["sentiment_label"]

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

In [ ]:
# create validation set
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_train_val,
)

In [ ]:
# verify data set split distribution
def summarize_split(name: str, labels: pd.Series) -> pd.DataFrame:
    counts = labels.value_counts()
    percentages = labels.value_counts(normalize=True).mul(100)

    return pd.DataFrame(
        {
            "split": name,
            "sentiment": counts.index,
            "count": counts.values,
            "percentage": percentages.values.round(2),
        }
    )


split_summary = pd.concat(
    [
        summarize_split("train", y_train),
        summarize_split("validation", y_val),
        summarize_split("test", y_test),
    ],
    ignore_index=True,
)

display(split_summary)

In [ ]:
#  Create the weighted TF-IDF Logistic Regression pipeline
baseline_pipeline = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                max_features=30_000,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1_000,
                solver="liblinear",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

In [ ]:
# train the model and track the training duration
training_start = perf_counter()

baseline_pipeline.fit(X_train, y_train)

training_time = perf_counter() - training_start

print(f"Training completed in {training_time:.2f} seconds.")

In [ ]:
# evaluate the model on the validation set and track the evaluation duration
inference_start = perf_counter()

val_predictions = baseline_pipeline.predict(X_val)

inference_time = perf_counter() - inference_start
average_inference_ms = (
    inference_time / len(X_val)
) * 1_000

In [ ]:
# Calculate metrics
val_accuracy = accuracy_score(y_val, val_predictions)

val_macro_f1 = f1_score(
    y_val,
    val_predictions,
    average="macro",
)

val_weighted_f1 = f1_score(
    y_val,
    val_predictions,
    average="weighted",
)

print(f"Validation accuracy:    {val_accuracy:.4f}")
print(f"Validation macro F1:    {val_macro_f1:.4f}")
print(f"Validation weighted F1: {val_weighted_f1:.4f}")
print(
    f"Average inference time: "
    f"{average_inference_ms:.4f} ms/review"
)

In [ ]:
# print the classification
print(
    classification_report(
        y_val,
        val_predictions,
        digits=4,
        zero_division=0,
    )
)

In [ ]:
# draw the confusion matrix
label_order = ["negative", "neutral", "positive"]

ConfusionMatrixDisplay.from_predictions(
    y_val,
    val_predictions,
    labels=label_order,
    normalize=None,
)

plt.title(
    "TF-IDF + Weighted Logistic Regression\n"
    "Validation Confusion Matrix"
)
plt.tight_layout()
plt.show()

In [ ]:
# draw normalsied confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_val,
    val_predictions,
    labels=label_order,
    normalize="true",
    values_format=".2f",
)

plt.title(
    "TF-IDF + Weighted Logistic Regression\n"
    "Normalized Validation Confusion Matrix"
)
plt.tight_layout()
plt.show()

In [ ]:
# inspect model coefficient
tfidf = baseline_pipeline.named_steps["tfidf"]
classifier = baseline_pipeline.named_steps["classifier"]

feature_names = tfidf.get_feature_names_out()

In [ ]:
top_features = get_top_features_per_class(
    classifier,
    feature_names,
    top_n=15,
)

for sentiment, feature_df in top_features.items():
    print(f"\nTop features for: {sentiment}")
    display(feature_df)

In [ ]:
# Save validation results
experiment_result = pd.DataFrame(
    [
        {
            "model_name": "tfidf_logistic_regression",
            "imbalance_strategy": "class_weight_balanced",
            "train_rows": len(X_train),
            "validation_rows": len(X_val),
            "accuracy": val_accuracy,
            "macro_f1": val_macro_f1,
            "weighted_f1": val_weighted_f1,
            "training_time_seconds": training_time,
            "average_inference_ms": average_inference_ms,
        }
    ]
)

result_path = (
    REPORTS_DIR /
    "tfidf_logistic_regression_validation.csv"
)

experiment_result.to_csv(result_path, index=False)

display(experiment_result)

In [ ]:
# Save the fitted pipeline for TF-IDF and Logistic Regression together
model_path = (
    ARTIFACTS_DIR /
    "tfidf_logistic_regression_pipeline.joblib"
)

joblib.dump(baseline_pipeline, model_path)

print(f"Saved model pipeline to: {model_path}")